# 🌊 Notebook 2 — Fine-tune MiDaS Depth Estimator for Potholes

**Model:** MiDaS DPT_Large  
**Output:** `midas_dpt_large_pothole_finetuned.pt`  
**Runtime:** GPU (T4 recommended)

## What this notebook does
1. Download PothRGBD dataset from Kaggle.
2. Build RGB ↔ depth pair list safely.
3. Fine-tune MiDaS decoder/head on paired data.
4. Save best checkpoint to Google Drive.

> Run all cells in order. This notebook stops early with clear errors if no training pairs are found.

In [ ]:
# ── 1) Check GPU ─────────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# ── 2) Install dependencies ──────────────────────────────────────────────────
!pip install -q --upgrade kagglehub opencv-python-headless matplotlib pyyaml
print('✅ Dependencies installed')

In [ ]:
# ── 3) Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/pothole_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print('✅ Save dir:', SAVE_DIR)

In [ ]:
# ── 4) Kaggle authentication ─────────────────────────────────────────────────
import os, json
from getpass import getpass

KAGGLE_USERNAME = input('Kaggle username: ').strip()
KAGGLE_KEY = getpass('Kaggle API key: ').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY
print('✅ Kaggle auth configured')

In [ ]:
# ── 5) Download PothRGBD ─────────────────────────────────────────────────────
import kagglehub
from pathlib import Path

pothrgbd_path = Path(kagglehub.dataset_download('mahyeks/pothrgbd-rgb-and-depth-images-of-potholes'))
print('✅ Downloaded to:', pothrgbd_path)

In [ ]:
# ── 6) Explore structure quickly ─────────────────────────────────────────────
import os

def tree(path, depth=2, indent=0, limit=15):
    if indent > depth:
        return
    try:
        items = sorted(os.listdir(path))
    except Exception:
        return
    for item in items[:limit]:
        full = os.path.join(path, item)
        print('  ' * indent + ('📁 ' if os.path.isdir(full) else '📄 ') + item)
        if os.path.isdir(full):
            tree(full, depth, indent + 1, limit)

print('=== PothRGBD ===')
tree(str(pothrgbd_path))

In [ ]:
# ── 7) Build RGB-depth pair manifest (robust) ───────────────────────────────
from pathlib import Path

IMG_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def list_images(root: Path):
    return [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_SUFFIXES]

def is_depth_image_path(p: Path):
    # IMPORTANT: only inspect local names, not full path (dataset name contains 'depth')
    stem = p.stem.lower()
    parent = p.parent.name.lower()
    return ('depth' in stem) or ('depth' in parent)

def normalize_key(name: str):
    s = name.lower()

    # Remove Roboflow hash chunk if present: ... .rf.<hash>
    if '.rf.' in s:
        s = s.split('.rf.')[0]

    # Remove common suffix tokens used in this dataset
    for token in ['_color_png', '_color', '_rgb', '_image', '_png', '_depth']:
        s = s.replace(token, '')

    return s

def score_root(root: Path):
    imgs = [p for p in list_images(root) if not is_depth_image_path(p)]
    depths = list(root.rglob('*.npy'))
    return len(imgs), len(depths)

# Try likely roots: root, children, grandchildren
candidate_roots = [pothrgbd_path]
try:
    children = [p for p in pothrgbd_path.iterdir() if p.is_dir()]
    candidate_roots.extend(children)
    for child in children:
        candidate_roots.extend([p for p in child.iterdir() if p.is_dir()])
except Exception:
    pass

# de-duplicate while preserving order
seen = set()
uniq_roots = []
for r in candidate_roots:
    s = str(r)
    if s not in seen:
        uniq_roots.append(r)
        seen.add(s)

best_root = pothrgbd_path
best_score = (-1, -1)
for root in uniq_roots:
    imgs_n, depths_n = score_root(root)
    if (imgs_n + depths_n) > (best_score[0] + best_score[1]):
        best_root = root
        best_score = (imgs_n, depths_n)

print(f'Using dataset root: {best_root}')

img_candidates = [p for p in list_images(best_root) if not is_depth_image_path(p)]
depth_candidates = list(best_root.rglob('*.npy'))

# Build depth index
# Example depth stem: 20250227_135438_depth -> key 20250227_135438
depth_index = {}
for d in depth_candidates:
    key = normalize_key(d.stem)
    depth_index.setdefault(key, []).append(d)

pairs = []
for img in img_candidates:
    key = normalize_key(img.stem)
    if key in depth_index:
        pairs.append((img, depth_index[key][0]))

print(f'RGB images found:   {len(img_candidates)}')
print(f'Depth npy found:    {len(depth_candidates)}')
print(f'Pairs constructed:  {len(pairs)}')

if len(img_candidates) == 0:
    print('\n[DEBUG] No image files found. Showing first-level structure of selected root:')
    try:
        for p in sorted(best_root.iterdir())[:30]:
            print(' -', p.name)
    except Exception:
        pass

if len(pairs) == 0:
    print('\n[DEBUG] Sample image stems:')
    for p in img_candidates[:5]:
        print(' -', p.stem, '->', normalize_key(p.stem))
    print('[DEBUG] Sample depth stems:')
    for p in depth_candidates[:5]:
        print(' -', p.stem, '->', normalize_key(p.stem))
    raise RuntimeError('No RGB-depth pairs found. Check selected root and filename patterns in debug output above.')

In [ ]:
# ── 8) Dataset + DataLoader ──────────────────────────────────────────────────
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

IMG_SIZE = 384
BATCH_SIZE = 6

class DepthPairDataset(Dataset):
    def __init__(self, pair_list):
        self.pairs = pair_list

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, depth_path = self.pairs[idx]

        img = cv2.imread(str(img_path))
        if img is None:
            raise RuntimeError(f'Failed to read image: {img_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = (img - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
        img = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()

        depth = np.load(str(depth_path)).astype(np.float32)
        depth = cv2.resize(depth, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        depth = np.nan_to_num(depth, nan=0.0, posinf=0.0, neginf=0.0)
        depth = np.clip(depth, 0.0, np.percentile(depth, 99.5) + 1e-6)
        depth = depth / (depth.max() + 1e-6)
        depth = torch.from_numpy(depth).float().unsqueeze(0)

        return img, depth

dataset = DepthPairDataset(pairs)
val_size = max(1, int(0.15 * len(dataset)))
train_size = len(dataset) - val_size
if train_size <= 0:
    raise RuntimeError('Not enough pairs for training. Need at least 2 pairs.')

train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train size: {len(train_ds)}, Val size: {len(val_ds)}')

In [ ]:
# ── 9) Load MiDaS model ──────────────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
midas = torch.hub.load('intel-isl/MiDaS', 'DPT_Large').to(device)
midas.train()

# Freeze most backbone layers, fine-tune lighter parts
for name, p in midas.named_parameters():
    if any(k in name.lower() for k in ['scratch', 'refinenet', 'output_conv', 'head']):
        p.requires_grad = True
    else:
        p.requires_grad = False

trainable = sum(p.numel() for p in midas.parameters() if p.requires_grad)
print('✅ MiDaS loaded on', device)
print('Trainable params:', trainable)

In [ ]:
# ── 10) Loss + optimizer ─────────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

class SILogLoss(nn.Module):
    def __init__(self, lam=0.85):
        super().__init__()
        self.lam = lam

    def forward(self, pred, target):
        pred = torch.clamp(pred, min=1e-6)
        target = torch.clamp(target, min=1e-6)
        g = torch.log(pred) - torch.log(target)
        return torch.sqrt((g ** 2).mean() - self.lam * (g.mean() ** 2))

criterion = SILogLoss().to(device)
optimizer = torch.optim.AdamW([p for p in midas.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
print('✅ Training components initialized')

In [ ]:
# ── 11) Fine-tune MiDaS ──────────────────────────────────────────────────────
import time
import torch.nn.functional as F

EPOCHS = 12
best_val = float('inf')
best_path = '/content/midas_dpt_large_pothole_finetuned.pt'

for epoch in range(1, EPOCHS + 1):
    midas.train()
    t0 = time.time()
    train_loss = 0.0

    for imgs, depth_gt in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        depth_gt = depth_gt.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred = midas(imgs)
            if pred.ndim == 3:
                pred = pred.unsqueeze(1)
            pred = F.interpolate(pred, size=depth_gt.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(pred, depth_gt)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    midas.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, depth_gt in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            depth_gt = depth_gt.to(device, non_blocking=True)

            pred = midas(imgs)
            if pred.ndim == 3:
                pred = pred.unsqueeze(1)
            pred = F.interpolate(pred, size=depth_gt.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(pred, depth_gt)
            val_loss += loss.item()

    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))
    dt = time.time() - t0
    print(f'Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | {dt:.1f}s')

    if val_loss < best_val:
        best_val = val_loss
        torch.save(midas.state_dict(), best_path)
        print(f'  ✅ Saved best checkpoint (val={best_val:.4f})')

print('\n✅ Fine-tuning complete')
print('Best model:', best_path)

In [ ]:
# ── 12) Quick qualitative check ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

midas.eval()
imgs, depth_gt = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    pred = midas(imgs)
    if pred.ndim == 3:
        pred = pred.unsqueeze(1)

pred_np = pred[0, 0].detach().cpu().numpy()
gt_np = depth_gt[0, 0].cpu().numpy()
img_np = imgs[0].detach().cpu().numpy().transpose(1, 2, 0)
img_np = (img_np * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_np = np.clip(img_np, 0, 1)

plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1); plt.imshow(img_np); plt.title('RGB'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(gt_np, cmap='magma'); plt.title('GT Depth (norm)'); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(pred_np, cmap='magma'); plt.title('Pred Depth'); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── 13) Save to Drive + download ─────────────────────────────────────────────
import shutil
from google.colab import files
from pathlib import Path

best_src = Path('/content/midas_dpt_large_pothole_finetuned.pt')
if not best_src.exists():
    raise FileNotFoundError('Best checkpoint not found. Run training cell first.')

best_dst = Path(SAVE_DIR) / 'midas_dpt_large_pothole_finetuned.pt'
shutil.copy(best_src, best_dst)
print('✅ Saved to Drive:', best_dst)

files.download(str(best_src))